# Front/Rear Brake Calculation Notes

This notebook estimates axle brake force, pedal force at tire lockup, and front/rear balance for the solar car brake system.

Annotations added in this pass explain what each block is doing, what assumptions are built in, and which values were sanity-checked by hand.


In [ ]:
# Imports.
# matplotlib is kept here in case brake-force curves are plotted later.
import numpy as np
import matplotlib.pyplot as plt


## Vehicle Constants

This block defines the wheelbase, track width, center-of-gravity location, and tire-road friction coefficient.

- `b` is the CG distance from the front axle.
- `c` is the CG distance from the rear axle.
- Static axle loads are computed from the standard longitudinal weight-distribution relations.
- This design case now uses `mu_s = 1.0` to represent the maximum `1g` braking / lockup scenario.
- All gravity-based mass/weight conversions now use `9.81 m/s^2` consistently.


In [ ]:
# Vehicle geometry and operating assumptions.
wb = 2.3  # m, wheelbase
tw = 1.55  # m, track width (not used yet in the current model)
x_cog = 31.37 * 0.0254  # m, CG distance from front axle
y_cog = 33.31 * 0.0254  # m, lateral CG location
z_cog = 19.90 * 0.0254  # m, CG height
mu_s = 1.0  # tire-road friction coefficient for the 1g max-braking case
g = 9.81  # m/s^2

w = 272 * g  # N, vehicle weight assuming 272 kg mass
m = w / g  # kg, recovered mass from weight above

# Longitudinal CG distances used in axle load transfer equations.
b = x_cog
c = wb - x_cog


## Hydraulic And Rotor Parameters

This converts diameters into piston areas and then combines hydraulic gain, pad friction, rotor effective radius, and tire radius into axle brake-force models.

Sanity check:
- Front hydraulic area ratio `cFa / mc1a = 2.56`
- Rear hydraulic area ratio `cRa / mc2a = 1.00`

The original notebook printed `cRa / mc1a`, which compared the rear caliper to the front master cylinder. That was likely not the intended rear-circuit diagnostic.


In [ ]:
# Brake-system dimensions.
mc1d = (5 / 8) * 0.0254  # m, front master cylinder diameter
mc2d = 1 * 0.0254  # m, rear master cylinder diameter
cFd = 1 * 0.0254  # m, front caliper piston diameter
cRd = 1 * 0.0254  # m, rear caliper piston diameter

# Piston areas.
mc1a = np.pi * (mc1d / 2) ** 2  # m^2, front master cylinder area
mc2a = np.pi * (mc2d / 2) ** 2  # m^2, rear master cylinder area
cFa = np.pi * (cFd / 2) ** 2  # m^2, front caliper piston area
cRa = np.pi * (cRd / 2) ** 2  # m^2, rear caliper piston area

print(cFa / mc1a)
print(cRa / mc2a)

mu_b = 0.35  # brake pad-disc friction coefficient
R_f = 10 * 0.0254 / 2  # m, front rotor radius
R_r = 16.75 * 0.0254 / 2  # m, rear rotor radius
r = 0.557 / 2  # m, tire radius

pedal_ratio = 4.35  # pedal box mechanical advantage


## Static Axle Loads

These are the no-braking axle loads from simple statics.

Checked values from the current inputs:
- Front axle total load `Wfs = 1743.92 N`, or `871.96 N` per front wheel
- Rear axle total load `Wrs = 924.40 N`


In [ ]:
Wfs = (c / wb) * w  # N, front axle static load
Wrs = (b / wb) * w  # N, rear axle static load

print(Wfs / 2)
print(Wrs)


## Brake Bias Model

- `beta` is the front fraction of total commanded brake force.
- `prop_valve_rear_factor` can reduce rear line effectiveness if a proportioning valve is introduced.
- `front_braking_force()` and `rear_braking_force()` convert pedal force directly to tire force at each axle.
- The pedal-force sweep is extended so the `1g` lockup pedal force remains inside the plotted/calculated range.


In [ ]:
prop_valve_rear_factor = 1
beta = 0.58  # front brake-force proportioning fraction

pf = np.arange(0, 700, 1)

def front_braking_force(pedal_force):
    # Four front pad faces are modeled in this expression.
    return 4 * pedal_force * pedal_ratio * (cFa / mc1a) * mu_b * (R_f / r)

def rear_braking_force(pedal_force):
    # Two rear pad faces are modeled in this expression.
    return 2 * pedal_force * pedal_ratio * (cRa / mc2a) * mu_b * (R_r / r)

def total_braking_force(pedal_force, front_bias=beta):
    rear_bias = 1 - front_bias
    return (
        4 * pedal_force * pedal_ratio * (cFa / mc1a) * mu_b * (R_f / r) * front_bias
        + 2
        * prop_valve_rear_factor
        * pedal_force
        * pedal_ratio
        * (cRa / mc2a)
        * mu_b
        * (R_r / r)
        * rear_bias
    )

front_bf = front_braking_force(pf)
rear_bf = rear_braking_force(pf)


## Front Lockup Threshold

Under braking, the front axle gains normal load from longitudinal weight transfer. These functions compute:
- the maximum front tire force before front lockup at a chosen deceleration level `a` in units of `g`
- the pedal force that would produce that front-axle lockup force with the current hydraulic setup and front bias

Checked values:
- Front lockup force: `2037.13 N` at `0.5g`, `2183.73 N` at `0.75g`, `2330.33 N` at `1.0g`
- Front lockup pedal force: `494.03 N` at `0.5g`, `529.58 N` at `0.75g`, `565.14 N` at `1.0g`


In [ ]:
def bf_lockup_front(a):
    return mu_s * (Wfs + (z_cog * w * a) / wb)

def pf_lockup_front(a):
    return (
        mu_s * (Wfs + (z_cog * w * a) / wb)
        / (4 * pedal_ratio * (cFa / mc1a) * mu_b * (R_f / r) * beta)
    )

half_g_front = bf_lockup_front(0.5)
three_fourths_g_front = bf_lockup_front(0.75)
one_g_front = bf_lockup_front(1)

half_g_front_pedal = pf_lockup_front(0.5)
three_fourths_g_front_pedal = pf_lockup_front(0.75)
one_g_front_pedal = pf_lockup_front(1)

print('max braking force for front: ', half_g_front, 'N (at 0.5g)')
print('max braking force for front: ', three_fourths_g_front, 'N (at 0.75g)')
print('max braking force for front: ', one_g_front, 'N (at 1g)')

print('max pedal force for front: ', half_g_front_pedal, 'N (at 0.5g)')
print('max pedal force for front: ', three_fourths_g_front_pedal, 'N (at 0.75g)')
print('max pedal force for front: ', one_g_front_pedal, 'N (at 1g)')


## 1g Design Case Summary

This is the main front-rotor design point for maximum braking / lockup.

With 2 front wheels and 2 front rotors:
- Total front braking force at `1g`: `2330.33 N`
- Front braking force per rotor / wheel: `1165.16 N`
- Total front-axle wheel torque at `1g`: `649.00 N*m`
- Torque per front rotor / wheel: `324.50 N*m`


In [ ]:
front_force_per_rotor_1g = one_g_front / 2
front_torque_total_1g = one_g_front * r
front_torque_per_rotor_1g = front_torque_total_1g / 2

print('front braking force at 1g (total front axle): ', one_g_front, 'N')
print('front braking force at 1g (per front rotor/wheel): ', front_force_per_rotor_1g, 'N')
print('front wheel torque at 1g (total front axle): ', front_torque_total_1g, 'N*m')
print('front wheel torque at 1g (per front rotor/wheel): ', front_torque_per_rotor_1g, 'N*m')


## Rear Lockup Threshold

The rear axle loses normal load during braking, so rear lockup force capacity decreases as deceleration rises.

Important correction:
- The original notebook used `bf_lockup_rear(0.1)` but labeled the result as `0g`.
- This has been changed to `0.5g` so the variable name, formula input, and print label all agree.

Checked values:
- Rear lockup force: `631.19 N` at `0.5g`, `484.59 N` at `0.75g`, `337.99 N` at `1.0g`
- Rear lockup pedal force: `646.15 N` at `0.5g`, `496.08 N` at `0.75g`, `346.00 N` at `1.0g`


In [ ]:
def bf_lockup_rear(a):
    return mu_s * (Wrs - (z_cog * w * a) / wb)

def pf_lockup_rear(a):
    return (
        mu_s * (Wrs - (z_cog * w * a) / wb)
        / (2 * prop_valve_rear_factor * pedal_ratio * (cRa / mc2a) * mu_b * (R_r / r) * (1 - beta))
    )

half_g_rear = bf_lockup_rear(0.5)
three_fourths_g_rear = bf_lockup_rear(0.75)
one_g_rear = bf_lockup_rear(1)

half_g_rear_pedal = pf_lockup_rear(0.5)
three_fourths_g_rear_pedal = pf_lockup_rear(0.75)
one_g_rear_pedal = pf_lockup_rear(1)

print('max braking force for rear: ', half_g_rear, 'N (at 0.5g)')
print('max braking force for rear: ', three_fourths_g_rear, 'N (at 0.75g)')
print('max braking force for rear: ', one_g_rear, 'N (at 1g)')

print('max pedal force for rear: ', half_g_rear_pedal, 'N (at 0.5g)')
print('max pedal force for rear: ', three_fourths_g_rear_pedal, 'N (at 0.75g)')
print('max pedal force for rear: ', one_g_rear_pedal, 'N (at 1g)')


## Bias / Lockup Interpretation

This compares the pedal force needed to lock the front axle versus the rear axle.

Interpretation rule:
- If rear lockup requires less pedal force than front lockup, the car is rear-biased and the rear tires lock first.
- If front lockup requires less pedal force, the system is front-biased.

Checked at `1.0g`:
- Front lockup pedal force: `565.14 N`
- Rear lockup pedal force: `346.00 N`
- Result: rear lockup happens first with the current `beta = 0.58`


In [ ]:
def will_front_wheels_lock_up_first(a):
    if pf_lockup_front(a) < pf_lockup_rear(a):
        return print('Front wheels will lock up before rear wheels')
    else:
        return print('Rear wheels will lock up before front wheels')

will_front_wheels_lock_up_first(1.0)


## Sanity Checks

- `Wfs / Wrs = 1.8866`, so the static car is front-heavy in this coordinate definition.
- The next two ratios simplify to `mu_s`, so they are mainly confirming that the lockup-force equation is of the form `F = mu * N`.
- At the front-lockup pedal force for `1.0g`, the front axle carries `2330.33 N` total braking force, which is `1165.16 N` per front rotor / wheel.
- The corresponding front-axle wheel torque is `649.00 N*m`, or `324.50 N*m` per front rotor / wheel.


In [ ]:
front_lockup_weight_ratio = (mu_s * (Wfs + (z_cog * w * 0.5 * g) / (wb * g))) / (Wfs + (z_cog * w * 0.5 * g) / (wb * g))
rear_lockup_weight_ratio = (mu_s * (Wrs - (z_cog * w * 0.5 * g) / (wb * g))) / (Wrs - (z_cog * w * 0.5 * g) / (wb * g))

print(Wfs / Wrs)
print(front_lockup_weight_ratio)
print(rear_lockup_weight_ratio)

a = total_braking_force(pf_lockup_front(0.5)) / m
print(total_braking_force(pf_lockup_front(0.5)))
print(a)
